# 1541. 平衡括号字符串的最少插入次数

匹配单位是一个 `(` 对应连续的 `))`。输入可能不平衡，目标是只插入字符，让插入次数最少。

[原文](https://leetcode.cn/problems/minimum-insertions-to-balance-a-parentheses-string/solution/1541-ping-heng-gua-hao-zi-fu-chuan-de-zu-i608/) · 整理于 2026-10-17。原代码保持原样；补充注释另放单元格。

## 右括号分组

每轮完整处理一组 `))`：缺左括号就补 `(`，缺第二个右括号就补 `)`；最后结算剩余左括号。

In [ ]:
# left_cnt：尚未匹配的左括号数；insert_cnt：已确定的插入次数。
# 访问 s[i+1] 前检查 i+1 < len(s)。
# 每轮不留下半组右括号，结束后每个 left_cnt 都缺两个右括号。

In [ ]:
class Solution:
    def minInsertions(self, s: str) -> int:
        left_cnt = insert_cnt = i = 0

        while i < len(s):
            if s[i] == "(":
                left_cnt += 1
                i += 1
            else:
                if left_cnt > 0:
                    left_cnt -= 1
                else:
                    insert_cnt += 1  # 补一个左括号

                if i + 1 < len(s) and s[i + 1] == ")":
                    i += 2  # 两个连续右括号组成一组
                else:
                    i += 1
                    insert_cnt += 1  # 补齐这一组的第二个右括号

        return insert_cnt + 2 * left_cnt

In [ ]:
for s, expected in [("())", 0), ("(", 2), ("))", 1), (")", 2), ("()(", 3), ("))(", 3), ("()()))", 3), ("(()))", 1), ("))())(", 3)]:
    assert Solution().minInsertions(s) == expected, s

## 逐字符维护 need

`need` 表示修补后的前缀还欠多少右括号。奇数表示存在半组 `))`；遇新 `(` 前必须先补齐这半组。

In [ ]:
# 遇左括号且 need 为奇数：先补一个右括号，need -= 1，再 need += 2。
# 遇右括号后 need == -1：补左括号；当前右括号已满足一半，令 need = 1。
# 扫描结束补 need 个右括号。

In [ ]:
class Solution:
    def minInsertions(self, s: str) -> int:
        need = 0
        insert_cnt = 0

        for ch in s:
            if ch == "(":
                if need % 2 == 1:
                    insert_cnt += 1
                    need -= 1
                need += 2
            else:
                need -= 1
                if need == -1:
                    insert_cnt += 1
                    need = 1

        return insert_cnt + need

In [ ]:
for s, expected in [("())", 0), ("(", 2), ("))", 1), (")", 2), ("()(", 3), ("))(", 3), ("()()))", 3), ("(()))", 1), ("))())(", 3)]:
    assert Solution().minInsertions(s) == expected, s

## 两种状态与踩坑

设未闭合左括号数为 L，已读半组右括号标记为 p∈{0,1}，则 `need = 2*L-p`。分组法每轮结束时 p=0；逐字符法要保存奇偶信息。

- 下方原始边界条件过严：`"())"` 会漏读已有的第二个右括号。
- 只返回 `insert_cnt` 会漏掉末尾欠账：`"("` 应返回 2。
- 1021 是一对一匹配且输入合法，不能直接把它的规则套到这里。

两版均为 O(n) 时间、O(1) 额外空间。局部缺失是必须插入的字符，优先复用已有 `))` 不会增加后续成本。

```python
if i < len(s) - 2:
    if s[i + 1] == ")":
        i += 2
```